In [ ]:
import json
import polars as pl
from plotnine import *

## Process mave db metadata

In [ ]:
with open('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/main.json') as f:
    d = json.load(f)

d

In [ ]:
d['experimentSets'][1]

In [ ]:
d['experimentSets'][0]['experiments'][0]['scoreSets'][0]

In [ ]:
d['experimentSets'][1]['experiments'][0]['scoreSets'][0]

In [ ]:
import pandas as pd

# Suppose your full list is called `data`
# Example: data = [ {...}, {...}, ... ]

rows = []

for item in d['experimentSets']:
    urn = item.get('urn')
    published_date = item.get('publishedDate')
    set_id = item.get('id')
    record_type = item.get('recordType')

    # Loop over experiments in this experiment set
    for exp in item.get('experiments', []):
        exp_title = exp.get('title')
        exp_short_desc = exp.get('shortDescription')
        exp_abstract = exp.get('abstractText')
        exp_method = exp.get('methodText')
        exp_urn = exp.get('urn')
        exp_creation_date = exp.get('creationDate')

        # Loop over scoreSets
        for score in exp.get('scoreSets', []):
            score_title = score.get('title')
            num_variants = score.get('numVariants')
            license_name = score.get('license', {}).get('longName')
            license_url = score.get('license', {}).get('link')
            
            target_genes = score.get('targetGenes', [])
            # Loop over target genes
            for gene in target_genes:
                gene_name = gene.get('name')

                organism_name = None
                target_sequence = gene.get('targetSequence')
                if target_sequence:
                    taxonomy = target_sequence.get('taxonomy')
                    if taxonomy:
                        organism_name = taxonomy.get('organismName')
            
                # Extract external IDs
                ensembl_id = None
                refseq_id = None
                uniprot_id = None

                for ext_id in gene.get('externalIdentifiers', []):
                    identifier = ext_id.get('identifier', {})
                    db_name = identifier.get('dbName', '').lower()
                    id_value = identifier.get('identifier')

                    if db_name == 'ensembl':
                        ensembl_id = id_value
                    elif db_name == 'refseq':
                        refseq_id = id_value
                    elif db_name == 'uniprot':
                        uniprot_id = id_value

                rows.append({
                    'set_urn': urn,
                    'set_published_date': published_date,
                    'set_id': set_id,
                    'record_type': record_type,

                    'experiment_title': exp_title,
                    'experiment_short_desc': exp_short_desc,
                    'experiment_abstract': exp_abstract,
                    'experiment_method': exp_method,
                    'experiment_urn': exp_urn,
                    'experiment_creation_date': exp_creation_date,

                    'score_title': score_title,
                    'num_variants': num_variants,
                    'license_name': license_name,
                    'license_url': license_url,

                    'target_gene': gene_name,
                    'ensembl_id': ensembl_id,
                    'refseq_id': refseq_id,
                    'uniprot_id': uniprot_id,
                    'organism_name': organism_name
                })

# Build DataFrame
mave_db = pl.DataFrame(rows)

# Extract gene symbols
mave_db = mave_db.with_columns(
    pl.col('target_gene').str.split(' ').list.get(0).alias('gene_symbol')
)

mave_db

In [ ]:
mave_db['organism_name'].value_counts().sort('count', descending=True)

In [ ]:
mave_db.filter(pl.col('organism_name')=='Homo sapiens')['target_gene'].value_counts().sort('count', descending=True)

In [ ]:
mave_db.filter(pl.col('organism_name')=='Homo sapiens')['gene_symbol'].value_counts().sort('count', descending=True)

## Check gene intersection

In [ ]:
gb_genes = pl.read_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass_1e6_coding_variants.parquet')
gb_genes

In [ ]:
gb_genes['region'].unique()


In [ ]:
mave_db.filter(pl.col('organism_name')=='Homo sapiens')['gene_symbol'].unique()